In [ ]:
import cv2
import math
import numpy as np
import easyocr
from collections import defaultdict
from ultralytics import YOLO

In [ ]:
# Entrenar modelo de reconocimiento de matrículas
model = YOLO('yolo26x.pt')
results = model.train(data='uc3m-lp.yaml', epochs=70, imgsz=640)

In [ ]:
model = YOLO('runs/detect/50epochs/weights/best.pt')

cap = cv2.VideoCapture("video.MP4")

while cap.isOpened():
    success, frame = cap.read()

    if success:
        results = model.track(frame, verbose=False)
        frame = results[0].plot()
        cv2.imshow("YOLO", frame)
        
        if cv2.waitKey(1) & 0xFF == ord("q"):
            break
    else:
        # Break the loop if the end of the video is reached
        break
cap.release()
cv2.destroyAllWindows()

In [ ]:
# Detección de personas, vehículos y matrículas SIN OCR

plates_model = YOLO('runs/detect/50epochs/weights/best.pt')
vehicle_model = YOLO('yolo26n.pt')

# Etiqueta de las distintas clases
classNames = ["person", "bicycle", "car", "motorbike", "aeroplane", "bus", "train", "truck"]

filename = "video.MP4"
cap = cv2.VideoCapture(filename)
track_history = defaultdict(None)

frame_width = int(cap.get(3))
frame_height = int(cap.get(4))
fps = cap.get(cv2.CAP_PROP_FPS)

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter('output.MP4', fourcc, fps, (frame_width, frame_height))

with open("results.csv", "w") as file:
    file.write("fotograma;tipo_objeto;confianza;identificador_tracking;x1;y1;x2;y2;matrícula_en_su_caso;confianza;mx1;my1;mx2;my2\n")

frame_num = 0
while cap.isOpened():
    # Read a frame from the video
    success, frame = cap.read()

    if success:
        frame_num += 1

        # Run YOLO inference on the frame
        results = vehicle_model.track(frame, persist=True, verbose=False, classes=[0,2,3,5,7], tracker="tracktrack.yaml")
        if results is not None:
            plates_res = plates_model(frame, verbose=False)
        else:
            plates_res = None

        #frame = plates_res[0].plot()

        # Para cada detección
        for r in results:
            boxes = r.boxes

            for box in boxes:
                # Contenedor
                x1, y1, x2, y2 = list(map(int, box.xyxy[0]))

                confidence = math.ceil((box.conf[0]*100))/100
                cls = int(box.cls[0])

                #Etiqueta de seguimiento
                if box.id is not None:
                    track_id = str(int(box.id[0].tolist()))
                    track_history[track_id] = cls
                else:
                    track_id = ''

                # Colores distintos para personas y vehículos
                if cls == 0:
                    R = 20
                    G = 180
                    B = 20
                else:
                    R = 100
                    G = 0
                    B = 200

                csv_string = f"{frame_num};{classNames[cls]};{confidence};{track_id};{x1};{y1};{x2};{y2}"

                # Dibuja el contenedor y clase
                cv2.rectangle(frame, (x1, y1), (x2, y2), (R, G, B), 3)
                cv2.putText(frame, track_id + ' ' + classNames[cls] , [x1, y1], cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)

                if cls != 0:
                    plate_found = False
                    for lpres in plates_res:
                        for plate in lpres.boxes:
                            lpx1, lpy1, lpx2, lpy2 = list(map(int, plate.xyxy[0]))
                            # Comprobar si la matrícula está dentro de la detección del vehículo
                            if lpx1 > x1 and lpx2 < x2 and lpy1 > y1 and lpy2 < y2:
                                plate_conf = math.ceil((plate.conf[0]*100))/100
                                cv2.rectangle(frame, (lpx1, lpy1), (lpx2, lpy2), (200, 50, 20), 3)
                                cv2.putText(frame, track_id + ' plate', [lpx1, lpy1], cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)
                                csv_string += f";true;{plate_conf};{lpx1};{lpy1};{lpx2};{lpy2}"
                                plate_found = True
                                break
                    if not plate_found:
                        csv_string += f";false;null;null;null;null;null"
                else:
                    csv_string += f";false;null;null;null;null;null"

                #print(csv_string)
                with open("results.csv", "a") as file:
                    file.write(csv_string + "\n")

        # Display the annotated frame
        out.write(frame)
        frame = cv2.resize(frame, (frame_width//2, frame_height//2), fx=0.5, fy=0.5, interpolation=cv2.INTER_LINEAR)
        cv2.imshow("YOLO Inference", frame)
        #cv2.imshow("YOLO Inference", results[0].plot())

        # Break the loop if 'q' is pressed
        if cv2.waitKey(1) & 0xFF == ord("q"):
            break
    else:
        # Break the loop if the end of the video is reached
        break

# Contar instancias de cada clase
with open("total.txt", "w") as file:
    for i, cls in enumerate(classNames):
        total = len(list(filter(lambda x: x[1] == i, track_history.items())))
        if total > 0:
            total_str = f"{cls}: {total}\n"
            print(total_str)
            file.write(total_str)

#results = model(filename, show=True)
cap.release()
out.release()
cv2.destroyAllWindows()

In [ ]:
# Detección de personas, vehículos y matrículas CON OCR (easyOCR)

plates_model = YOLO('runs/detect/50epochs/weights/best.pt')
vehicle_model = YOLO('yolo26n.pt')
ocr_reader = easyocr.Reader(['es', 'en'])
#plate_chars = "ABCDEFGHIJKLMNOPQRSTUVWXYZ1234567890-"

# Etiqueta de las distintas clases
classNames = ["person", "bicycle", "car", "motorbike", "aeroplane", "bus", "train", "truck"]

filename = "video.MP4"
cap = cv2.VideoCapture(filename)
track_history = defaultdict(None)

frame_width = int(cap.get(3))
frame_height = int(cap.get(4))
fps = cap.get(cv2.CAP_PROP_FPS)

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter('output.MP4', fourcc, fps, (frame_width, frame_height))

with open("results.csv", "w") as file:
    file.write("fotograma;tipo_objeto;confianza;identificador_tracking;x1;y1;x2;y2;matrícula_en_su_caso;confianza;mx1;my1;mx2;my2\n")

frame_num = 0
while cap.isOpened():
    # Read a frame from the video
    success, frame = cap.read()

    if success:
        frame_num += 1

        # Run YOLO inference on the frame
        results = vehicle_model.track(frame, persist=True, verbose=False, classes=[0,2,3,5,7], tracker="tracktrack.yaml")
        if results is not None:
            plates_res = plates_model(frame, verbose=False)
        else:
            plates_res = None

        #frame = plates_res[0].plot()

        # Para cada detección
        for r in results:
            boxes = r.boxes

            for box in boxes:
                # Contenedor
                x1, y1, x2, y2 = list(map(int, box.xyxy[0]))

                confidence = math.ceil((box.conf[0]*100))/100
                cls = int(box.cls[0])

                #Etiqueta de seguimiento
                if box.id is not None:
                    track_id = str(int(box.id[0].tolist()))
                    track_history[track_id] = cls
                else:
                    track_id = ''

                # Colores distintos para personas y vehículos
                if cls == 0:
                    R = 20
                    G = 180
                    B = 20
                else:
                    R = 100
                    G = 0
                    B = 200

                csv_string = f"{frame_num};{classNames[cls]};{confidence};{track_id};{x1};{y1};{x2};{y2}"

                # Dibuja el contenedor y clase
                cv2.rectangle(frame, (x1, y1), (x2, y2), (R, G, B), 3)
                cv2.putText(frame, track_id + ' ' + classNames[cls] , [x1, y1], cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)

                if cls != 0:
                    plate_found = False
                    for lpres in plates_res:
                        for plate in lpres.boxes:
                            lpx1, lpy1, lpx2, lpy2 = list(map(int, plate.xyxy[0]))
                            # Comprobar si la matrícula está dentro de la detección del vehículo
                            if lpx1 > x1 and lpx2 < x2 and lpy1 > y1 and lpy2 < y2:
                                plate_conf = math.ceil((plate.conf[0]*100))/100
                                plate_frame = frame[lpy1:lpy2, lpx1:lpx2].copy()
                                cv2.rectangle(frame, (lpx1, lpy1), (lpx2, lpy2), (200, 50, 20), 3)
                                cv2.putText(frame, track_id + ' plate', [lpx1, lpy1], cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)
                                csv_string += f";true;{plate_conf};{lpx1};{lpy1};{lpx2};{lpy2}"
                                plate_frame = cv2.resize(plate_frame, (600, 200))
                                cv2.imshow("Id: " + str(track_id), plate_frame)
                                """ocr_res = ocr_reader.recognize(plate_frame)
                                if len(ocr_res) > 0:
                                    print(ocr_res)"""
                                plate_found = True
                                break
                    if not plate_found:
                        csv_string += f";false;null;null;null;null;null"
                else:
                    csv_string += f";false;null;null;null;null;null"

                #print(csv_string)
                with open("results.csv", "a") as file:
                    file.write(csv_string + "\n")

        # Display the annotated frame
        out.write(frame)
        frame = cv2.resize(frame, (frame_width//2, frame_height//2), fx=0.5, fy=0.5, interpolation=cv2.INTER_LINEAR)
        cv2.imshow("YOLO Inference", frame)
        #cv2.imshow("YOLO Inference", results[0].plot())

        # Break the loop if 'q' is pressed
        if cv2.waitKey(1) & 0xFF == ord("q"):
            break
    else:
        # Break the loop if the end of the video is reached
        break

# Contar instancias de cada clase
with open("total.txt", "w") as file:
    for i, cls in enumerate(classNames):
        total = len(list(filter(lambda x: x[1] == i, track_history.items())))
        if total > 0:
            total_str = f"{cls}: {total}\n"
            print(total_str)
            file.write(total_str)

#results = model(filename, show=True)
cap.release()
out.release()
cv2.destroyAllWindows()


Neither CUDA nor MPS are available - defaulting to CPU. Note: This module is much faster with a GPU.


In [ ]:
cv2.destroyAllWindows()